# Cầu nối NumPy → PyTorch (45 phút)
    Mục tiêu: cùng một batch Digits 1×8×8 và cùng weight cho cùng forward, CE, gradient và một bước SGD.
    Ví dụ nhỏ này được giải đầy đủ để tập trung hiểu autograd; CNN project quần áo vẫn do bạn tự viết.
    Mở `../appendix/A_PYTORCH.md` khi cần tra cứu.

In [ ]:
import os, sys
from pathlib import Path
# Chạy tại root repo hoặc trong cnn/, nlp/, teacher/notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'shared/data.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Giải nén/clone toàn bộ repo, rồi chuyển thư mục tới nơi có shared/. Xem CLOUD.md.')
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import torch
import matplotlib.pyplot as plt
from shared.data import load_data, describe, normalized_images
from shared.baselines import run_baselines
from shared.utils import load_config, read_json
from shared.engine import fit, make_loader, evaluate, reload_model, predict_image, predict_text
from shared.metrics import plot_confusion, plot_history, show_errors
from shared.protocol import register_protocol, lock_selection, final_test
from shared.kiem_tra import check_model, check_padding
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(4)
DATA_ROOT = Path(os.environ.get('AI4B_DATA_ROOT', 'data'))
OUTPUT_ROOT = Path(os.environ.get('AI4B_OUTPUT_ROOT', 'runs/my_project'))
SMOKE = os.environ.get('AI4B_SMOKE') == '1'  # Chỉ kiểm kỹ thuật, không dùng để báo cáo benchmark.
if SMOKE: OUTPUT_ROOT = OUTPUT_ROOT / 'smoke'
print('PyTorch', torch.__version__, '| device:', DEVICE)
print('Data:', DATA_ROOT.resolve(), '| Output:', OUTPUT_ROOT.resolve(), '| smoke:', SMOKE)

## 1. Shape trước số học (0–8 phút)
    Conv(1→8,3,pad1) → ReLU → AvgPool2 → Flatten128 → Linear10.
    Viết shape sau từng lớp với batch=8. AvgPool chia gradient cho 4; ReLU chặn gradient khi input âm.
    Câu hỏi: nếu đổi AvgPool thành MaxPool, backward phải thay phần nào?

In [ ]:
from sklearn.datasets import load_digits
x = load_digits().images[:8, None] / 16
print(x.shape, x.min(), x.max())
plt.imshow(x[0,0], cmap='gray');

## 2. Đọc forward và backward NumPy (8–22 phút)
    Mở ô nguồn bên dưới: `patches` trích lân cận 3×3; `einsum` tính tổng tích.
    Theo dấu `dz` từ softmax + CE về Linear, AvgPool, ReLU, rồi Conv.
    Đọc từng shape và chỉ ra vị trí chain rule. Không cần học thuộc einsum.
    Vì sao gradient logits trừ 1 ở nhãn thật rồi chia batch size?

In [ ]:
"""CNN Digits NumPy với backward đầy đủ, để đối chiếu autograd CPU float64."""
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.datasets import load_digits
import torch
from torch import nn


def numpy_loss_grad(x, y, weights):
    wc, bc, wl, bl = weights
    n = len(x)
    xp = np.pad(x, ((0,0), (0,0), (1,1), (1,1)))
    patches = sliding_window_view(xp, (3,3), axis=(2,3))
    conv = np.einsum('nchwij,ocij->nohw', patches, wc) + bc[None,:,None,None]
    relu = np.maximum(conv, 0)
    pool = relu.reshape(n,8,4,2,4,2).mean(axis=(3,5))
    flat = pool.reshape(n,128)
    logits = flat @ wl.T + bl
    stable = logits - logits.max(1, keepdims=True)
    prob = np.exp(stable); prob /= prob.sum(1, keepdims=True)
    loss = -(stable[np.arange(n),y] - np.log(np.exp(stable).sum(1))).mean()
    dz = prob.copy(); dz[np.arange(n),y] -= 1; dz /= n
    dwl, dbl = dz.T @ flat, dz.sum(0)
    dpool = (dz @ wl).reshape(n,8,4,4)
    drelu = np.repeat(np.repeat(dpool, 2, axis=2), 2, axis=3) / 4
    dconv = drelu * (conv > 0)
    dwc = np.einsum('nohw,nchwij->ocij', dconv, patches)
    dbc = dconv.sum(axis=(0,2,3))
    dxp = np.zeros_like(xp)
    for i in range(3):
        for j in range(3):
            dxp[:,:,i:i+8,j:j+8] += np.einsum('nohw,oc->nchw', dconv, wc[:,:,i,j])
    return logits, loss, [dwc, dbc, dwl, dbl], dxp[:,:,1:-1,1:-1]


def run_bridge():
    torch.manual_seed(7)
    digits = load_digits()
    x = digits.images[:8,None].astype(np.float64) / 16
    y = digits.target[:8]
    model = nn.Sequential(nn.Conv2d(1,8,3,padding=1), nn.ReLU(), nn.AvgPool2d(2),
                          nn.Flatten(), nn.Linear(128,10)).double().cpu()
    params = list(model.parameters())
    weights = [p.detach().numpy().copy() for p in params]
    logits, loss, grads, dx = numpy_loss_grad(x, y, weights)
    tx = torch.tensor(x, requires_grad=True)
    torch_logits = model(tx)
    torch_loss = nn.functional.cross_entropy(torch_logits, torch.tensor(y))
    torch_loss.backward()
    differences = {'forward': float(np.max(np.abs(logits-torch_logits.detach().numpy()))),
                   'loss': abs(loss-torch_loss.item()), 'input_gradient': float(np.max(np.abs(dx-tx.grad.numpy())))}
    for i, (p, g) in enumerate(zip(params, grads)):
        differences[f'parameter_gradient_{i}'] = float(np.max(np.abs(g-p.grad.numpy())))
        np.testing.assert_allclose(g, p.grad.numpy(), atol=1e-6, rtol=1e-6)
    lr = .05
    updated = [w-lr*g for w,g in zip(weights,grads)]
    torch.optim.SGD(model.parameters(), lr=lr).step()
    differences['one_update'] = max(float(np.max(np.abs(w-p.detach().numpy()))) for w,p in zip(updated,params))
    assert max(differences.values()) < 1e-6
    return {'input_shape': list(x.shape), 'loss': float(loss), 'max_absolute_errors': differences}

## 3. Đối chiếu autograd (22–35 phút)
    `run_bridge` sao cùng weight vào hai cách tính. CPU float64 giúp đọc sai số số học.
    Quan sát lỗi forward, loss, gradient từng parameter, gradient input và một update; tất cả <1e-6.
    Thử đổi nhầm dấu gradient NumPy rồi chạy: assertion phải bắt được lỗi. Khôi phục trước khi tiếp.
    Vì sao phải so weight giống nhau thay vì chỉ đặt cùng tên model?

In [ ]:
result = run_bridge()
print(result)

## 4. Từ update sang optimizer (35–45 phút)
    Ghép `zero_grad`, `loss.backward`, `optimizer.step` với bước NumPy tương ứng.
    Câu hỏi: gọi backward hai lần trước zero_grad sẽ tạo gradient gì? `detach` có tác dụng gì?
    Bài tự kiểm: vẽ chuỗi forward → loss → gradient → update; giải thích từng bước bằng lời.
    Tiếp theo: mở project CNN, tự hoàn thiện train step và dùng Adam theo cấu hình.